ACOLITE Example 2: OLCI processing

This notebook allows you to run ACOLITE processing on Google Colab. The required dependencies are installed and the latest version is cloned from GitHub (https://github.com/acolite/acolite).

In this example, Sentinel-3 scenes are retrieved from CDSE and hence requires your credentials to be set up. Download and edit the credentials.txt file, and upload it to the files in the current runtime. Ancillary data can be retrieved if EarthData credentials are set up, but are not essential for this example.

Author: Quinten Vanhellemont, RBINS, quinten.vanhellemont@naturalsciences.be

Written 2026-05-11 for VUB Oceans and Lakes course

In [ ]:
# general imports
import os, sys

# install dependencies for acolite
# for colab: assume if netCDF4 is installed the other dependencies are as well
try:
  import netCDF4
except:
  !pip install numpy matplotlib scipy gdal pyproj scikit-image pyhdf pyresample netcdf4 h5py requests pygrib cartopy

In [ ]:
# clone acolite repo if it does not exist
if not os.path.exists('acolite'):
  !git clone --depth 1 https://github.com/acolite/acolite

In [ ]:
# add acolite to path and import
sys.path.append('acolite')
import acolite as ac

In [ ]:
## we will use the CDSE API for download
## and credentials need to be imported
## load credentials and set as env variables
credentials = ac.shared.import_config('credentials.txt')
for k in credentials: os.environ[k] = credentials[k]

In [ ]:
## location and date of interest
## you can use the Copernicus browser to find images
## https://browser.dataspace.copernicus.eu/
region_name = 'Oostende'
station_lon = 2.90
station_lat = 51.25
date = '2026-05-09'

In [ ]:
## before we used a point to find scenes,
##roi = 'POINT ({} {})'.format(station_lon, station_lat)
## now we will use a box as we will be processing a larger ROI
roi = ac.shared.limit.station_check(station_lon, station_lat, 50, 'km')['limit']

In [ ]:
## use ACOLITE API to query CDSE
## CDSE credentials not needed for query
urls, scenes = ac.api.cdse.query(roi = roi,
                                 collection = "SENTINEL-3", timeliness = 'NT',
                                 start_date = date, end_date = date)
## print the results
for scene in scenes:
  print(scene)

In [ ]:
## here we will download the OLCI scene to the runtime environment
## this will take longer than the GCS transfer

## make input directory
if not os.path.exists('Input/'):
  os.makedirs('Input/')

## get copy of the L1C bundles
local_scenes = []
for si, scene in enumerate(scenes):
  local_scene = 'Input/{}'.format(scene)
  if os.path.exists(local_scene):
    print('We have {}'.format(local_scene))
    local_scenes.append(local_scene)
    continue

  ## use ACOLITE API to download the scenes from CDSE
  ## CDSE needed for download
  if not os.path.exists(local_scene):
    local_scene = ac.api.cdse.download(urls[si], scenes = scenes[si], output = 'Input/')[0]

  ## add to local scenes list
  if os.path.exists(local_scene):
    print('Downloaded {}'.format(local_scene))
    local_scenes.append(local_scene)

In [ ]:
# acolite settings
settings = {## basic input and output settings
            ## local_scenes list is used as returned from the API download function
            'inputfile': local_scenes,
            'output': 'Output/',

            ## set up region of interest centred on lon/lat, with box size in km
            'region_name': region_name,
            'station_lon': station_lon,
            'station_lat': station_lat,
            'station_box_size': 50,
            'station_box_units': 'km',

            ## merge scenes
            'merge_tiles': True,

            ## add a limit buffer and reproject data
            'limit_buffer': 0.1,
            'output_projection':  True,
            'reproject_before_ac':  True,

            ## disable ancillary data retrieval, set to True if EarthData credentials are set
            'ancillary_data': False,

            ## compress NetCDF files
            'netcdf_compression': True,

            ## delete L1R output NetCDF
            'l1r_delete_netcdf': True,
            ## don't output L1R RGBs
            'l1r_rgb_keys': [],
            ## delete ACOLITE text outputs
            'delete_acolite_run_text_files': True,
            }

In [ ]:
## run acolite with settings dict
ret = ac.acolite.acolite_run(settings)